# Péndulo simple, parte 1

> Una masa puntual de $1\ \text{kg}$ en un péndulo simple se deja caer con un ángulo de
> $85\degree$ respecto a la vertical. La cuerda (que es rígida) tiene una longitud de
> $30\ \text{m}$. ¿Cuántas oscilaciones habrá completado después de $24\ \text{h}$?

A través de mecánica vectorial o de métodos de mecánica analítica se puede mostrar que
$$
  \ddot{\phi}(t) = - \frac{g}{L} \sin{\phi(t)}
$$
donde $\phi$ es el ángulo del péndulo (respecto a la vertical),
$L$ es la longitud del péndulo y $g$ es la aceleración gravitatoria.
Para muchos fines la aproximación de ángulos pequeños
$$
  \sin{\phi} \approx \phi
$$
es apropiada, y encontramos que el periodo es aproximadamente
$$
  T \approx 2 \pi \sqrt{\frac{L}{g}}.
$$

In [ ]:
import math

In [ ]:
g = 9.80665  # m/s^2
m = 1  # kg
L = 30  # m
phi_max = math.radians(85)
period = 24 * 3600  # 24h * 3600 s/h

In [ ]:
T_aprox = 2 * math.pi * math.sqrt(L / g)
print(T_aprox)

In [ ]:
oscill_aprox = math.floor(period / T_aprox)  # la función piso
print(f"El número de oscilaciones es aprox. {oscill_aprox}")

Por supuesto, $85\degree$ *definitivamente* no es un ángulo pequeño.
Es seguro que esta aproximación no es muy buena.

Escribí un pequeño módulo que calcula el valor real del periodo, `pendulum.py`.
Comparemos con nuestra aproximación.

In [ ]:
import pendulum as pdreal

T_real = pdreal.period(L=L, phi_max=phi_max)
print(T_real)

Estoy ocultando la implementación de `pendulum.py` a propósito.
Quiero mostrar que la aproximación de ángulos pequeñas falla, pero las matemáticas
necesarias no son importantes para lo que buscamos con estas notas, y la implementación
usa herramientas que no hemos introducido.

In [ ]:
oscill_real = math.floor(period / T_real)
print(f"El número de oscilaciones es de hecho {oscill_real}")
oscill_aprox_error = (oscill_aprox - oscill_real) / oscill_real
print(f"La aproximación tiene un error de {100 * oscill_aprox_error:.1f}%")

Abandonemos entonces la aproximación de ángulos pequeños.
¿Qué hacemos? Partir de la ecuación diferencial de segundo grado
es posible, pero gracias a la conservación de energía podemos mostrar que
$$
  \dot{\phi}^2 = \frac{2 g}{L} \left[ \cos{\phi(t)} - \cos{\phi_{\max}} \right].
$$
Podemos plantear cuidadosamente una integral para el periodo (con cuidado de que
integrar una región en la que la velocidad no cambie de signo)
$$
  T = 4 \sqrt{\frac{L}{2 g}} \int_{-\phi_{\max}}^{0}
    \frac{\mathrm{d} \phi}{\sqrt{\cos{\phi} - \cos{\phi_{\max}}}}
$$
_i.e._ un cuarto de la oscilación completa.
Esta forma de la integral es un poco inconveniente porque el integrando tiene a infinito
en $\phi \to -\phi_{\max}^{-}$, pero un cambio de variable
$$
  \sin{\varphi} \equiv \frac{1}{k} \sin{\frac{\phi}{2}}, \quad
    k = \sin{\frac{\phi_{\max}}{2}}
$$
nos permite mostrar que
$$
  T = 4 \sqrt{\frac{L}{g}} \int_{-\pi/2}^{0}
    \frac{\mathrm{d} \varphi}{\sqrt{1 - k^2 \sin^2{\varphi}}}.
$$
Este integrando no causa problemas en este intervalo.

:::{tip}
No te preocupes si este desarrollo no es obvio para ti. Es más importante lo que vamos a
hacer a continuación.
:::

A pesar del cambio de variable, esta no es una integral con solución bonita,
así que tendremos que usar _métodos numéricos_. Especificamente, debemos hacer una
integración numérica.

:::{note} Definición
Métodos numéricos
: Algoritmos que aproximan soluciones a problemas matemáticos.

  A diferencia de evaluaciones analíticas como las que haría una matemática o incluso un
  programa de álgebra computacional como Mathematica, los métodos numéricos realizan
  un número (usualmente grande, siempre finito) de operaciones aritméticas. Esto es útil
  cuando computar una función requiere de un número infinito de pasos (o al menos muy grande)
  y no existe una expresión equivalente fácil de calcular a mano.

  El costo de usar estos métodos es que (casi) todos los resultados son aproximaciones.
  Usualmente a mayor número de operaciones mayor es la precisión resultante. La precisión
  de un buen algoritmo crece de forma relativamente lenta respecto al número de
operaciones.
:::

Entonces, la forma ingenua para resolver la integral es *discretizar* la definición
de la integral como sumas de Riemann,
$$
  \sum f(x_j) \, {\Delta x_j} \to \int f(x) \, \mathrm{d}x.
$$

Seleccionemos una partición para nuestro intervalo de integración.
Hay maneras muy sofisticadas de seleccionar una partición para integración numérica,
pero para este ejemplo simple usemos $N$ puntos igualmente espaciados en el
intervalo $[x_i, x_f]$.
Nuestra partición es
$$
  P = \{ x_1, x_2, \ldots, x_n \}
$$
donde, para $ j \in \{1, \ldots, n\} $,

$$
\begin{align*}
  x_j &= x_i + j (x_f - x_i)/N \\
      &= x_i + j {\Delta x}
\end{align*}
$$

Al hacer esto definimos el tamaño del paso (_step-size_)
$ {\Delta x} \equiv (x_f - x_i)/N $,
la separación entre puntos de la partición.

Así,

$$
\begin{align*}
  \int_{x_i}^{x_f} f(x) \, \mathrm{d}x
    &\approx \sum_{j=1}^{N} f(x_j) \, {\Delta x} \\
    &= {\Delta x} \sum_{j=1}^{N} f(x_j)
\end{align*}
$$

Implementemos esto en código:

In [ ]:
def simple_integration(function, xstart, xend, N=100):
    """
    Calcula la integral numérica de una función de una variable con una partición
    de N puntos igualmente espaciados.

    Parámetros
    ----------
    function  : función a integral
    xstart    : extremo inicial de integración
    xend      : extremo final de integración
    N         : número de puntos en la partición
                (por defecto 100)

    Regresa
    -------
    I         : Valor de la integral
    """

    Delta = (xend - xstart) / N

    # recordemos que los índices empiezan en 0
    partition = [xstart + j * Delta for j in range(1, N + 1)]

    I = Delta * math.fsum(function(x) for x in partition)

    return I

¿Cómo podemos determinar el valor de $N$? De nuevo, hay métodos más sofisticados,
pero adoptemos aquí un criterio sencillo: una *precisión relativa* de $10^{-4} \%$.

In [ ]:
def simpleint_reltol(
    function, xstart, xend, Nmin=20, Nmax=2000, reltol=1.0e-6, skipn=10
):
    """
    Calcula la integral numérica de una función de una variable, aumentando N
    hasta que la diferencia relativa entre estimaciones sucesivas sea menor a
    `reltol`.

    Ver `simple_integration` para más detalles.

    Parámetros
    ----------
    function  : función a integrar
    xstart    : extremo inicial de integración
    xend      : extremo final de integración
    Nmin      : número inicial de puntos en la partición
                (por defecto 10)
    Nmax      : número máximo de puntos en la partición
                (por defecto 1000)
    reltol    : tolerancia relativa entre estimaciones sucesivas
                (por defecto 1.e-6)
    skipn     : incremento en el número de puntos entre iteraciones sucesivas
                (por defecto 10)

    Regresa
    -------
    Inew      : Valor de la integral con la N que alcanza `reltol`
                (o la última estimación calculada, si no se alcanza `reltol`
                antes de Nmax)
    """

    Iref = simple_integration(function, xstart, xend, N=Nmin)
    for n in range(Nmin + skipn, Nmax + 1, skipn):
        Inew = simple_integration(function, xstart, xend, N=n)
        if abs((Inew - Iref) / Iref) < reltol:
            print(n)
            return Inew
        Iref = Inew

    print("¡No hubo convergencia!")

    return Iref

In [ ]:
print("Integrando sin(x) de 0 a pi con N = 100")
print(simple_integration(math.sin, 0.0, math.pi))
print("Integrando con N adaptable")
print(simpleint_reltol(math.sin, 0.0, math.pi))

Este es un primer, ingeniuo intento de implimentar *convergencia*, el criterio
que nos permite determinar cuántas operaciones realizar en función de la precisión
deseada.

:::{warning} ¡No uses este código en producción!
No lo copies y pegues en tus proyectos.
Escribí estas funciones con fin ilustrativo.
Son lentas y no tienen mecanismos para tratar errores o estimar errores absolutos.
:::

*Finalmente*, usemos estas funciones para calcular el periodo del péndulo.

In [ ]:
def numeric_period(L, phi_max, g=g):
    """
    Calcula numéricamente el periodo de un péndulo simple, sin aproximaciones,
    usando integración simple.

    Parámetros
    ----------
    L       :   Longitud del péndulo [m]
    phi_max :   ángulo máximo [rad]
    g       :   aceleración gravitatoria [m/s^2]
                (por defecto, la estandar)

    Regresa
    -------
    T       :   Periodo del péndulo [s]
    """

    k = math.sin(phi_max / 2.0)

    def integrand(phi):
        return 1.0 / math.sqrt(1 - (k**2) * math.sin(phi) ** 2)

    T = 4 * math.sqrt(L / g) * simpleint_reltol(integrand, -math.pi / 2, 0.0)

    return T

In [ ]:
print("Valor de N para la partición y periodo obtenido.")
T_numeric = numeric_period(L, phi_max)
print(T_numeric, "\n")

oscill_numeric = math.floor(period / T_numeric)
print(f"Ahora predecimos {oscill_numeric} oscilaciones.\n")

print("Recordemos nuestro resultado 'real'")
print(oscill_real)

oscill_numeric_error = (oscill_numeric - oscill_real) / oscill_real
print(f"Nuestro error es ahora {100*oscill_numeric_error:.2f}%")
print("¡Mucho mejor!")